### Gold: Perfil de Risco do Cliente

In [0]:
from pyspark.sql import functions as F

silver_application = spark.table("credit_risk_pipeline.silver.application")
silver_bureau = spark.table("credit_risk_pipeline.silver.bureau")
silver_previous = spark.table("credit_risk_pipeline.silver.previous_application")
silver_installments = spark.table("credit_risk_pipeline.silver.installments_payments")

# Resumo do bureau: de várias linhas por cliente para uma só
agg_bureau = (
    silver_bureau
    .groupBy("SK_ID_CURR")
    .agg(
        F.count("*").alias("qtd_creditos_bureau"),
        F.sum(F.when(F.col("CREDIT_ACTIVE") == "Active", 1).otherwise(0)).alias("qtd_creditos_ativos_bureau"),
        F.avg("CREDIT_DAY_OVERDUE").alias("atraso_medio_dias_bureau"),
        F.sum("AMT_CREDIT_SUM_OVERDUE").alias("valor_total_overdue_bureau")
    )
)

# Resumo das propostas anteriores na própria instituição
agg_previous = (
    silver_previous
    .groupBy("SK_ID_CURR")
    .agg(
        F.count("*").alias("qtd_propostas_anteriores"),
        F.sum(F.when(F.col("NAME_CONTRACT_STATUS") == "Approved", 1).otherwise(0)).alias("qtd_propostas_aprovadas"),
        F.avg("AMT_APPLICATION").alias("valor_medio_solicitado_anterior")
    )
    .withColumn("taxa_aprovacao_anterior", F.col("qtd_propostas_aprovadas") / F.col("qtd_propostas_anteriores"))
)

# Resumo do comportamento de pagamento
agg_installments = (
    silver_installments
    .filter(F.col("parcela_paga") == True)
    .groupBy("SK_ID_CURR")
    .agg(
        F.count("*").alias("qtd_parcelas_pagas"),
        F.avg("atraso_dias").alias("atraso_medio_dias_pagamento"),
        F.avg((F.col("atraso_dias") > 0).cast("int")).alias("pct_parcelas_atrasadas")
    )
)

# Junta o perfil do cliente com os três resumos
gold_perfil_risco_cliente = (
    silver_application
    .join(agg_bureau, on="SK_ID_CURR", how="left")
    .join(agg_previous, on="SK_ID_CURR", how="left")
    .join(agg_installments, on="SK_ID_CURR", how="left")
    .fillna({
        "qtd_creditos_bureau": 0,
        "qtd_creditos_ativos_bureau": 0,
        "valor_total_overdue_bureau": 0,
        "qtd_propostas_anteriores": 0,
        "qtd_propostas_aprovadas": 0,
        "qtd_parcelas_pagas": 0
    })
)

gold_perfil_risco_cliente.write.mode("overwrite").saveAsTable("credit_risk_pipeline.gold.perfil_risco_cliente")

print(f"Tabela gold.perfil_risco_cliente criada com {gold_perfil_risco_cliente.count()} linhas e {len(gold_perfil_risco_cliente.columns)} colunas.")

### Gold: Indicadores de Risco

In [0]:
from pyspark.sql import functions as F

gold_perfil = spark.table("credit_risk_pipeline.gold.perfil_risco_cliente")

# Calcula os cortes de tercil (33% e 66%) pra renda e valor de crédito
limites_renda = gold_perfil.approxQuantile("AMT_INCOME_TOTAL", [0.33, 0.66], 0.01)
limites_credito = gold_perfil.approxQuantile("AMT_CREDIT", [0.33, 0.66], 0.01)

gold_com_segmentos = (
    gold_perfil
    .withColumn(
        "faixa_etaria",
        F.when(F.col("idade_anos") < 30, "até 29")
         .when(F.col("idade_anos") < 45, "30 a 44")
         .when(F.col("idade_anos") < 60, "45 a 59")
         .otherwise("60 ou mais")
    )
    .withColumn(
        "faixa_renda",
        F.when(F.col("AMT_INCOME_TOTAL") <= limites_renda[0], "baixa")
         .when(F.col("AMT_INCOME_TOTAL") <= limites_renda[1], "média")
         .otherwise("alta")
    )
    .withColumn(
        "faixa_valor_credito",
        F.when(F.col("AMT_CREDIT") <= limites_credito[0], "baixo")
         .when(F.col("AMT_CREDIT") <= limites_credito[1], "médio")
         .otherwise("alto")
    )
)

gold_indicadores_risco = (
    gold_com_segmentos
    .groupBy("faixa_etaria", "faixa_renda", "faixa_valor_credito")
    .agg(
        F.count("*").alias("qtd_clientes"),
        F.sum("TARGET").alias("qtd_inadimplentes"),
        F.avg("TARGET").alias("taxa_inadimplencia"),
        F.sum("AMT_CREDIT").alias("exposicao_total_credito"),
        F.sum(F.when(F.col("TARGET") == 1, F.col("AMT_CREDIT")).otherwise(0)).alias("exposicao_em_risco")
    )
)

gold_indicadores_risco.write.mode("overwrite").saveAsTable("credit_risk_pipeline.gold.indicadores_risco")

print(f"Tabela gold.indicadores_risco criada com {gold_indicadores_risco.count()} linhas e {len(gold_indicadores_risco.columns)} colunas.")